In [ ]:
from glob import glob
from os import path
from datetime import datetime as dt, timedelta
import xarray as xr
import numpy as np
from scipy.stats import linregress
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib import dates as mdates
from goes2go.data import goes_nearesttime
from cartopy import crs as ccrs
from cartopy import feature as cfeat
from pyxlma import coords
import cmweather
import warnings
from metpy.units import units
from metpy import calc as mpcalc
from metpy import plots as mpplots
from geopandas import read_file

In [ ]:
sbf_fineline_ex_dt = dt(2022, 7, 14, 21, 50, 0)
goes_data = goes_nearesttime(sbf_fineline_ex_dt, satellite='goes16', product='ABI', return_as='xarray')
goes_data['TrueColor'] = goes_data.rgb.TrueColor()
imkw = goes_data.rgb.imshow_kwargs
imkw.pop('transform')

In [ ]:
radar_filepaths = sorted(glob(f'/Volumes/LtgSSD/nexrad_zarr/{sbf_fineline_ex_dt.strftime('%B').upper()}/{sbf_fineline_ex_dt.strftime('%Y%m%d')}/*'))
radar_times = np.array([dt.strptime(path.basename(f), 'KHGX%Y%m%d_%H%M%S_V06_grid.zarr') for f in radar_filepaths])
closest_radar_idx = np.argmin(np.abs(radar_times - sbf_fineline_ex_dt))
closest_radar_path = radar_filepaths[closest_radar_idx]
radar_data = xr.open_dataset(closest_radar_path, engine='zarr').isel(time=0, nradar=0)
tpcs_coords = coords.TangentPlaneCartesianSystem(ctrLat=float(radar_data.radar_latitude.data), ctrLon=float(radar_data.radar_longitude.data), ctrAlt=float(radar_data.radar_altitude.data))
geosys = coords.GeographicSystem()
x2d, y2d = np.meshgrid(radar_data.x.data, radar_data.y.data)
ecef_coords = tpcs_coords.toECEF(x2d.flatten(), y2d.flatten(), np.full_like(x2d, 1000).flatten())
radar_lon, radar_lat, _ = lla = geosys.fromECEF(*ecef_coords)
radar_lon.shape = x2d.shape
radar_lat.shape = y2d.shape

In [ ]:
madis_file = path.join(path.sep, 'Volumes', 'LtgSSD', 'sfcdata_madis', sbf_fineline_ex_dt.strftime('%Y%m%d_*'))
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')
    madis_ds = xr.open_mfdataset(madis_file, engine='netcdf4', chunks='auto', coords='minimal', concat_dim='recNum', combine='nested', compat='override')
madis_ds = madis_ds.where(((madis_ds.longitude <= radar_lon.max()) & (madis_ds.longitude >= radar_lon.min()) & (madis_ds.latitude <= radar_lat.max()) & (madis_ds.latitude >= radar_lat.min())).compute(), drop=True)
dims_to_rm = list(madis_ds.dims)
dims_to_rm.remove('recNum')
madis_ds = madis_ds.drop_dims(dims_to_rm)
madis_ds_temp = madis_ds.temperature.data
madis_ds_temp_qc = madis_ds.temperatureQCR.data

madis_ds_dew = madis_ds.dewpoint.data
madis_ds_dew_qc = madis_ds.dewpointQCR.data

madis_ds_time = madis_ds.observationTime.data
madis_ds_lat = madis_ds.latitude.data
madis_ds_lon = madis_ds.longitude.data

madis_ds_invalid = np.zeros_like(madis_ds_temp, dtype=bool)
madis_ds_invalid[((madis_ds_temp_qc != 0) | (madis_ds_dew_qc != 0) | np.isnan(madis_ds_temp) | np.isnan(madis_ds_dew)).compute()] = True

madis_ds_temp[madis_ds_invalid] = np.nan
madis_ds_temp = madis_ds_temp.compute()
madis_ds_dew[madis_ds_invalid] = np.nan
madis_ds_dew = madis_ds_dew.compute()
madis_ds_time[madis_ds_invalid] = np.datetime64('NaT')
madis_ds_time = madis_ds_time.astype('datetime64[s]').compute()
madis_ds_lat[madis_ds_invalid] = np.nan
madis_ds_lat = madis_ds_lat.compute()
madis_ds_lon[madis_ds_invalid] = np.nan
madis_ds_lon = madis_ds_lon.compute()

lower_time_bound = np.array([sbf_fineline_ex_dt-timedelta(hours=1)]).astype('datetime64[s]')[0]
upper_time_bound = np.array([sbf_fineline_ex_dt]).astype('datetime64[s]')[0]
before = (madis_ds.observationTime <= upper_time_bound).compute()
after = (madis_ds.observationTime >= lower_time_bound).compute()
rec_to_consider = madis_ds.where((before & after), drop=True)
df = rec_to_consider[['observationTime', 'stationId']].to_dataframe()
latest_records_idx = df.groupby('stationId')['observationTime'].idxmax()
latest_obs = rec_to_consider.sel(recNum=latest_records_idx)
dir = latest_obs.windDir.data.compute() * units.degrees
spd = (latest_obs.windSpeed.data.compute() * units.meter / units.second).to(units.knots)
u, v = mpcalc.wind_components(spd, dir)

In [ ]:
sbf_interp = read_file(f'sam_polyline/{sbf_fineline_ex_dt.strftime("%Y-%m-%d_interpolated.json")}').set_index('index', drop=True)
poly_i_want = sbf_interp[sbf_interp.index == radar_data.time.data.astype('datetime64[s]').astype(dt)]

In [ ]:
sbf_fineline_ex_fig = plt.figure(figsize=(10, 10))
sbf_fineline_ex_axs = sbf_fineline_ex_fig.subplots(2, 2, subplot_kw={'projection' : ccrs.LambertConformal()})
[ax.set_extent([-96, -93, 28, 31], crs=ccrs.PlateCarree()) for ax in sbf_fineline_ex_axs.flatten()]
sbf_fineline_ex_axs[0, 0].imshow(goes_data['TrueColor'].data, **imkw, transform=goes_data.rgb.crs)
sbf_fineline_ex_axs[0, 0].set_title(f'GOES-16 True Color\n{goes_data.time_bounds.min().astype("datetime64[s]").data.astype(dt).item().strftime("%Y-%m-%d %H:%M:%S")}')
rdr_ref = sbf_fineline_ex_axs[0, 1].pcolormesh(radar_lon, radar_lat, radar_data.reflectivity.sel(z=1000), vmin=-10, vmax=80, cmap='ChaseSpectral', transform=ccrs.PlateCarree(), rasterized=True)
sbf_fineline_ex_axs[0, 1].set_title(f'KHGX 1km AGL Radar Reflectivity\n{radar_data.time.data.astype("datetime64[s]").astype(dt).item().strftime("%Y-%m-%d %H:%M:%S")}')
sbf_fineline_ex_fig.colorbar(rdr_ref, ax=sbf_fineline_ex_axs[0, 1], orientation='vertical', label='Reflectivity (dBZ)')
[ax.add_feature(cfeat.COASTLINE) for ax in sbf_fineline_ex_axs.flatten()]
stations = mpplots.StationPlot(sbf_fineline_ex_axs[1, 0], latest_obs.longitude, latest_obs.latitude, clip_on=True, transform=ccrs.PlateCarree(), fontsize=6)
stations.plot_barb(u, v, sizes={"emptybarb" : 0}, zorder=2)
sbf_fineline_ex_axs[1, 0].set_title(f'MADIS Surface Wind Barbs (kt)\n{sbf_fineline_ex_dt.strftime("%Y-%m-%d %H:%M")}')

sbf_fineline_ex_axs[1, 1].set_facecolor('tab:red')
poly_i_want.plot(ax=sbf_fineline_ex_axs[1, 1], color='tab:blue', linewidth=1, transform=ccrs.PlateCarree())
sbf_fineline_ex_axs[1, 1].set_title('Subjectively Analyzed Seabreeze Front\n(Red continental airmass, Blue maritime airmass)')

sbf_fineline_ex_fig.suptitle(f'Seabreeze Identification Example')
sbf_fineline_ex_fig.tight_layout()

sbf_fineline_ex_fig.savefig(f'./thesis_figs/seabreeze_bdy_example.pdf')

In [ ]:
pyrcel_ccn_path = '/Volumes/LtgSSD/arm-ccn-fix/sam_pyrcel_out.csv'
pyrcel_ccn = pd.read_csv(pyrcel_ccn_path, parse_dates=['timestamp']).set_index('timestamp', drop=True)
pyrcel_ccn

In [ ]:
aerosol_dataset_paths_arm = glob('/Volumes/LtgSSD/arm-ccn-avg/*.nc')
aos_obs = xr.open_mfdataset(aerosol_dataset_paths_arm, engine='netcdf4', combine='by_coords').load()

In [ ]:
aos_zero_point_four = aos_obs.isel(time=((aos_obs.supersaturation_calculated >= 0.4) & (aos_obs.supersaturation_calculated <= 0.6)).compute())
aos_zero_point_six = aos_obs.isel(time=((aos_obs.supersaturation_calculated >= 0.6) & (aos_obs.supersaturation_calculated <= 0.8)).compute())

In [ ]:
timeseries_fig, (ax4, ax6) = plt.subplots(2, 1, figsize=(10, 5))
ax4.scatter(aos_zero_point_four.time.data, aos_zero_point_four.N_CCN.data, s=1, c='tab:blue', label='DoE CCN-200', alpha=0.8)
ax4.scatter(pyrcel_ccn.index, pyrcel_ccn['0.4SS'], s=1, c='tab:red', label='pyrcel', alpha=0.8)
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax4.set_title('0.4% supersaturation')
ax4.set_ylabel('CCN Concentration (cm$^{-3}$)')
ax4.set_xlabel('Time')
ax4.legend()
ax6.scatter(aos_zero_point_six.time.data, aos_zero_point_six.N_CCN.data, s=1, c='tab:blue', label='DoE CCN-200', alpha=0.8)
ax6.scatter(pyrcel_ccn.index, pyrcel_ccn['0.6SS'], s=1, c='tab:red', label='pyrcel', alpha=0.8)
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax6.set_title('0.6% supersaturation')
ax6.set_ylabel('CCN Concentration (cm$^{-3}$)')
ax6.set_xlabel('Time')
ax6.legend()
timeseries_fig.suptitle('CCN Comparison between ARM CCN-200 instrument and HTDMA inversion + pyrcel simulation')
timeseries_fig.tight_layout()
timeseries_fig.savefig(f'./thesis_figs/ccn_ts.pdf')

In [ ]:
hourly_obs_point_four = aos_zero_point_four.resample(time='1h').mean()
hourly_obs_point_six = aos_zero_point_six.resample(time='1h').mean()
hourly_obs_point_four = hourly_obs_point_four.isel(time=hourly_obs_point_four.time.isin(pyrcel_ccn.index))
hourly_obs_point_six = hourly_obs_point_six.isel(time=hourly_obs_point_six.time.isin(pyrcel_ccn.index))
pyrcel_ccn4 = pyrcel_ccn['0.4SS'].reindex(hourly_obs_point_four.time.data, method='nearest')
pyrcel_ccn6 = pyrcel_ccn['0.6SS'].reindex(hourly_obs_point_six.time.data, method='nearest')

In [ ]:
hourly_obs_point_four

In [ ]:
comparison_fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 6))
four_handle = ax1.scatter(hourly_obs_point_four.N_CCN.data, pyrcel_ccn4, s=1, c=hourly_obs_point_four.supersaturation_calculated)
ax1.set_xlabel('CCN-200 Concentration (cm$^{-3}$)')
ax1.set_ylabel('pyrcel CCN Concentration (cm$^{-3}$)')
ax1.plot([0, 4500], [0, 4500], color='tab:gray', linestyle='--', linewidth=1, alpha=0.5, label='Ideal (1:1)')
four_reg = linregress(hourly_obs_point_four.N_CCN.data[~np.isnan(hourly_obs_point_four.N_CCN.data) & ~np.isnan(pyrcel_ccn4)],
                      pyrcel_ccn4[~np.isnan(hourly_obs_point_four.N_CCN.data) & ~np.isnan(pyrcel_ccn4)])
ax1.plot([0, 4500], [four_reg.intercept, four_reg.intercept + four_reg.slope * 4500], color='tab:red', linestyle='-', linewidth=1, alpha=0.5, label='Linear Fit')
ax1.set_title(f'0.4% supersaturation\ny={four_reg.slope:.2f}x + {four_reg.intercept:.1f}, r$^2={four_reg.rvalue**2:.2f}$')
ax1.legend()
comparison_fig.colorbar(four_handle, ax=ax1, label=f'CCN-200 actual supersaturation (%)', orientation='horizontal')
six_handle = ax2.scatter(hourly_obs_point_six.N_CCN.data, pyrcel_ccn6, s=1, c=hourly_obs_point_six.supersaturation_calculated)
ax2.set_xlabel('CCN-200 Concentration (cm$^{-3}$)')
ax2.set_ylabel('pyrcel CCN Concentration (cm$^{-3}$)')
six_reg = linregress(hourly_obs_point_six.N_CCN.data[~np.isnan(hourly_obs_point_six.N_CCN.data) & ~np.isnan(pyrcel_ccn6)],
                      pyrcel_ccn6[~np.isnan(hourly_obs_point_six.N_CCN.data) & ~np.isnan(pyrcel_ccn6)])
ax2.plot([0, 8000], [0, 8000], color='tab:gray', linestyle='--', linewidth=1, alpha=0.5, label='Ideal (1:1)')
ax2.plot([0, 8000], [six_reg.intercept, six_reg.intercept + six_reg.slope * 8000], color='tab:red', linestyle='-', linewidth=1, alpha=0.5, label='Linear Fit')
ax2.set_title(f'0.6% supersaturation\ny={six_reg.slope:.2f}x + {six_reg.intercept:.1f}, r$^2={six_reg.rvalue**2:.2f}$')
ax2.legend()
comparison_fig.colorbar(six_handle, ax=ax2, label='CCN-200 actual supersaturation (%)', orientation='horizontal')
comparison_fig.suptitle('CCN Comparison between ARM CCN-200 instrument and HTDMA inversion + pyrcel simulation')
comparison_fig.tight_layout()

In [ ]:
comparison_fig.savefig(f'./thesis_figs/ccn_comparison.pdf')

In [ ]:
seabreeze_ds = xr.open_dataset('/Volumes/LtgSSD/tobac_saves/tobac_Save_20220622/seabreeze-obs.zarr')

In [ ]:
seabreeze_progression = np.full(seabreeze_ds.seabreeze.shape, np.datetime64('NaT'), dtype='datetime64[s]')
for i, t in enumerate(seabreeze_ds.time.data):
    this_seabreeze = seabreeze_ds.seabreeze.isel(time=i)
    this_progression = seabreeze_progression[i, :, :]
    this_progression[this_seabreeze == -1] = t

In [ ]:
sbf_time_reached = (seabreeze_progression - np.nanmin(seabreeze_progression)).astype(float)
sbf_time_reached[sbf_time_reached < 0] = np.nan
sbf_time_reached = np.nanmin(sbf_time_reached, axis=0)

In [ ]:
sbf_prog_fig = plt.figure(figsize=(10, 7))
sbf_prog_ax = plt.axes(projection=ccrs.PlateCarree())
prog_handle = sbf_prog_ax.pcolormesh(seabreeze_ds.lon.data, seabreeze_ds.lat.data, sbf_time_reached.T/3600, cmap='plasma', rasterized=True)
sbf_prog_ax.add_feature(mpplots.USCOUNTIES.with_scale('5m'))
sbf_prog_fig.colorbar(prog_handle, ax=sbf_prog_ax, orientation='vertical', label='Time of seabreeze front passage (UTC hour)')
sbf_prog_fig.suptitle('Seabreeze Progression, 2022-06-22')
sbf_prog_fig.tight_layout()
sbf_prog_fig.savefig(f'./thesis_figs/seabreeze_progression.pdf')

In [ ]:
feature_example_ds = xr.open_dataset('/Volumes/LtgSSD/tobac_saves/tobac_Save_20220801/seabreeze-obs.zarr')
feature_i_want = 19375#18539
ex_feat_time = feature_example_ds.sel(feature=feature_i_want)
this_time = ex_feat_time.feature_time.data.astype('datetime64[s]')
this_time_idx = np.argmin(np.abs(ex_feat_time.time.data - this_time))
this_time = this_time.astype(dt).item() - timedelta(seconds=1)
previous_time_idx = this_time_idx - 1
previous_time = feature_example_ds.time.data[previous_time_idx].astype('datetime64[s]').astype(dt)
ex_feat_time = ex_feat_time.isel(time=this_time_idx)

In [ ]:
radar_ds = xr.open_dataset(f'/Volumes/LtgSSD/nexrad_zarr/AUGUST/20220801/KHGX20220801_{this_time.strftime('%H%M%S')}_V06_grid.zarr')
radar_ds = radar_ds.isel(time=0, nradar=0)
meltlayer = 4900
freezelayer = 10800
closest_to_melt_layer = np.argmin(np.abs(radar_ds.z.data - meltlayer))
closest_to_freeze_layer = np.argmin(np.abs(radar_ds.z.data - freezelayer))

mixed_phase = radar_ds.isel(z=slice(closest_to_melt_layer, closest_to_freeze_layer))
zdr_volume = ((mixed_phase.differential_reflectivity >= 1.0) & (mixed_phase.reflectivity >= 10)).sum(dim='z').astype(float).data
zdr_volume[zdr_volume == 0] = np.nan
kdp_volume = ((mixed_phase.KDP_CSU >= 0.75) & (mixed_phase.reflectivity >= 10)).sum(dim='z').astype(float).data
kdp_volume[kdp_volume == 0] = np.nan

lightning_filepaths = glob(f'/Volumes/LtgSSD/{int(this_time.strftime("%m"))}/6sensor_minimum/LYLOUT_{this_time.strftime("%y%m%d")}*.nc')
lightning_filepath = lightning_filepaths[0]
lightning = xr.open_dataset(lightning_filepath)
lightning_data_at_time = lightning.sel(grid_time=slice(previous_time, this_time))
flash_mask = (lightning_data_at_time.flash_time_start.data > np.datetime64(previous_time)) & (lightning_data_at_time.flash_time_end.data < np.datetime64(this_time))
event_mask = (lightning_data_at_time.event_time.data > np.datetime64(previous_time)) & (lightning_data_at_time.event_time.data < np.datetime64(this_time))
lightning_data_at_time = lightning_data_at_time.isel(number_of_flashes=flash_mask, number_of_events=event_mask)

feature_segm_mask = ex_feat_time.segmentation_mask == feature_i_want
xs_of_feature = ex_feat_time.x.data * feature_segm_mask.max(dim='y')
xs_of_feature[~feature_segm_mask.max(dim='y')] = np.nan
feature_min_x = xs_of_feature.min().item()
feature_max_x = xs_of_feature.max().item()
ys_of_feature = ex_feat_time.y.data * feature_segm_mask.max(dim='x')
ys_of_feature[~feature_segm_mask.max(dim='x')] = np.nan
feature_min_y = ys_of_feature.min().item()
feature_max_y = ys_of_feature.max().item()

In [ ]:
seg_mask_mask = ex_feat_time.segmentation_mask.data != feature_i_want
seg_mask_mask = seg_mask_mask.astype(float)
seg_mask_mask[seg_mask_mask == 0] = np.nan

In [ ]:
ex_fig, axes = plt.subplots(2, 2, figsize=(8, 9.6))
refl_handle = axes[0, 0].pcolormesh(ex_feat_time.lon, ex_feat_time.lat, radar_ds.reflectivity.max(dim='z'), cmap='ChaseSpectral', vmin=-10, vmax=80, rasterized=True)
axes[0, 0].scatter(lightning_data_at_time.flash_center_longitude, lightning_data_at_time.flash_center_latitude, c=lightning_data_at_time.flash_id, cmap='tab20', s=20, linewidths=0.5, edgecolors='black', label='Flash Center')
axes[0, 0].pcolormesh(ex_feat_time.lon, ex_feat_time.lat, seg_mask_mask, cmap='Greys', alpha=0.75, rasterized=True)
axes[0, 0].set_title(f'KHGX Composite Z, tobac segmentation mask,\nflash centers: {this_time.strftime("%Y-%m-%d %H:%M:%S")} UTC')
axes[0, 0].set_xlabel('Longitude (°E)')
axes[0, 0].set_ylabel('Latitude (°N)')
ex_fig.colorbar(refl_handle, ax=axes[0, 0], orientation='horizontal', label='Reflectivity (dBZ)')
axes[0, 1].pcolormesh(ex_feat_time.x/1000, ex_feat_time.y/1000, radar_ds.reflectivity.max(dim='z'), cmap='ChaseSpectral', vmin=-10, vmax=80, rasterized=True)
axes[0, 1].pcolormesh(ex_feat_time.x/1000, ex_feat_time.y/1000, seg_mask_mask, cmap='Greys', alpha=0.75, rasterized=True)
axes[0, 1].set_xlim(feature_min_x/1000, feature_max_x/1000)
axes[0, 1].set_ylim(feature_min_y/1000, feature_max_y/1000)
axes[0, 1].scatter(lightning_data_at_time.flash_ctr_x/1000, lightning_data_at_time.flash_ctr_y/1000, c=lightning_data_at_time.flash_id, cmap='tab20b', marker='*', s=250, label='Flash Center')
flash_handle = axes[0, 1].scatter(lightning_data_at_time.event_x/1000, lightning_data_at_time.event_y/1000, c=lightning_data_at_time.event_parent_flash_id, cmap='tab20b', s=10, linewidths=0.5, edgecolors='k', label='VHF Events')
axes[0, 1].set_title(f'KHGX Composite Z, tobac segmentation mask,\nflash centers and VHF events')
axes[0, 1].set_xlabel('X (East-west distance from radar, km)')
axes[0, 1].set_ylabel('Y (North-south distance from radar, km)')
ex_fig.colorbar(flash_handle, ax=axes[0, 1], orientation='horizontal', label='Flash ID')
zdr_handle = axes[1, 0].pcolormesh(ex_feat_time.x/1000, ex_feat_time.y/1000, zdr_volume, cmap='tab20', vmin=0, vmax=9, rasterized=True)
axes[1, 0].pcolormesh(ex_feat_time.x/1000, ex_feat_time.y/1000, seg_mask_mask, cmap='Greys_r', alpha=0.75, rasterized=True)
axes[1, 0].set_xlim(feature_min_x/1000, feature_max_x/1000)
axes[1, 0].set_ylim(feature_min_y/1000, feature_max_y/1000)
axes[1, 0].set_title(f'ZDR Volume, tobac segmentation mask')
axes[1, 0].set_xlabel('X (East-west distance from radar, km)')
axes[1, 0].set_ylabel('Y (North-south distance from radar, km)')
ex_fig.colorbar(zdr_handle, ax=axes[1, 0], orientation='horizontal', label='ZDR Volume (grid cell count)')
kdp_handle = axes[1, 1].pcolormesh(ex_feat_time.x/1000, ex_feat_time.y/1000, kdp_volume, cmap='tab20', vmin=0, vmax=9, rasterized=True)
axes[1, 1].pcolormesh(ex_feat_time.x/1000, ex_feat_time.y/1000, seg_mask_mask, cmap='Greys_r', alpha=0.75, rasterized=True)
axes[1, 1].set_xlim(feature_min_x/1000, feature_max_x/1000)
axes[1, 1].set_ylim(feature_min_y/1000, feature_max_y/1000)
axes[1, 1].set_title(f'KDP Volume, tobac segmentation mask')
axes[1, 1].set_xlabel('X (East-west distance from radar, km)')
axes[1, 1].set_ylabel('Y (North-south distance from radar, km)')
ex_fig.colorbar(kdp_handle, ax=axes[1, 1], orientation='horizontal', label='KDP Volume (grid cell count)')
ex_fig.suptitle(f'Feature ID: {feature_i_want} | Flash Count: {ex_feat_time.feature_flash_count.data.item():d} | ZDR Volume: {ex_feat_time.feature_zdrvol.data.item()}'+r' km$^{3}$'+f'\nKDP Volume: {ex_feat_time.feature_kdpvol.data.item()}'+r' km$^{3}$ '+f'| Max Z: {ex_feat_time.feature_maxrefl.data.item():.1f} dBZ | Footprint Area: {ex_feat_time.feature_area.data.item()*.25*.25}'+r' km$^{2}$')
ex_fig.tight_layout()


In [ ]:
ex_fig.savefig(f'./thesis_figs/pol_example.pdf')

In [ ]:
repr_sounding_ds = xr.open_dataset('/Volumes/LtgSSD/tobac_saves/tobac_Save_20220714/seabreeze-obs.zarr')

In [ ]:
repr_sounding_ds.continental_u_profile
repr_sounding_ds.continental_v_profile
repr_sounding_ds.continental_pressure_profile
repr_sounding_ds.continental_msl_profile
repr_sounding_ds.continental_dewpoint_profile
repr_sounding_ds.continental_temperature_profile

In [ ]:
launch_times = ['2022-07-14T00:00:00.000000000', '2022-07-14T01:10:00.000000000',
 '2022-07-14T01:30:00.000000000', '2022-07-14T01:40:00.000000000',
 '2022-07-14T02:30:00.000000000', '2022-07-14T05:29:00.000000000',
 '2022-07-14T06:00:00.000000000', '2022-07-14T06:40:00.000000000',
 '2022-07-14T11:10:00.000000000', '2022-07-14T11:29:00.000000000',
 '2022-07-14T12:20:00.000000000', '2022-07-14T13:00:00.000000000',
 '2022-07-14T13:40:00.000000000', '2022-07-14T13:50:00.000000000',
 '2022-07-14T14:30:00.000000000', '2022-07-14T14:40:00.000000000',
 '2022-07-14T14:50:00.000000000', '2022-07-14T15:10:00.000000000',
 '2022-07-14T15:30:00.000000000', '2022-07-14T17:30:00.000000000',
 '2022-07-14T18:40:00.000000000', '2022-07-14T20:20:00.000000000',
 '2022-07-14T20:20:00.000000000', '2022-07-14T20:30:00.000000000',
 '2022-07-14T21:30:00.000000000', '2022-07-14T22:30:00.000000000',
 '2022-07-14T23:29:00.000000000']
launch_times = np.array(launch_times).astype('datetime64[s]')
launch_times

In [ ]:
repr_sounding_fig, axes = plt.subplots(6, 1, figsize=(8, 10))
pres_handle = axes[0].pcolormesh(repr_sounding_ds.time, repr_sounding_ds.vertical_levels, repr_sounding_ds.continental_pressure_profile.T, cmap='viridis', rasterized=True)
axes[0].set_title('Pressure')
repr_sounding_fig.colorbar(pres_handle, ax=axes[0], orientation='vertical', label='hPa')
temp_handle = axes[1].pcolormesh(repr_sounding_ds.time, repr_sounding_ds.vertical_levels, repr_sounding_ds.continental_temperature_profile.T, cmap='turbo', rasterized=True)
axes[1].set_title('Temperature')
repr_sounding_fig.colorbar(temp_handle, ax=axes[1], orientation='vertical', label='°C')
dew_handle = axes[2].pcolormesh(repr_sounding_ds.time, repr_sounding_ds.vertical_levels, repr_sounding_ds.continental_dewpoint_profile.T, cmap='BrBG', rasterized=True)
axes[2].set_title('Dewpoint')
repr_sounding_fig.colorbar(dew_handle, ax=axes[2], orientation='vertical', label='°C')
u_handle = axes[3].pcolormesh(repr_sounding_ds.time, repr_sounding_ds.vertical_levels, repr_sounding_ds.continental_u_profile.T, cmap='coolwarm', rasterized=True,
                              vmin=-1*np.max(np.abs(repr_sounding_ds.continental_u_profile)), vmax=np.max(np.abs(repr_sounding_ds.continental_u_profile)))
axes[3].set_title('East Wind Component')
repr_sounding_fig.colorbar(u_handle, ax=axes[3], orientation='vertical', label='m/s')
v_handle = axes[4].pcolormesh(repr_sounding_ds.time, repr_sounding_ds.vertical_levels, repr_sounding_ds.continental_v_profile.T, cmap='coolwarm', rasterized=True,
                              vmin=-1*np.max(np.abs(repr_sounding_ds.continental_v_profile)), vmax=np.max(np.abs(repr_sounding_ds.continental_v_profile)))
axes[4].set_title('North Wind Component')
repr_sounding_fig.colorbar(v_handle, ax=axes[4], orientation='vertical', label='m/s')
msl_handle = axes[5].pcolormesh(repr_sounding_ds.time, repr_sounding_ds.vertical_levels, repr_sounding_ds.continental_msl_profile.T/1000, cmap='plasma', rasterized=True, vmin=0, vmax=25)
axes[5].set_title('Height')
repr_sounding_fig.colorbar(msl_handle, ax=axes[5], orientation='vertical', label='km MSL')
repr_sounding_fig.supylabel('Interpolated Vertical Levels')
[ax.yaxis.set_ticks(np.arange(0, 2001, 500)) for ax in axes]
[ax.set_ylim(0, 2000) for ax in axes]
[ax.tick_params(axis='x', labelbottom=False) for ax in axes[:-1]]
[ax.vlines(launch_times, 0, 2000, color='k', linestyle='--') for ax in axes]
axes[5].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
repr_sounding_fig.supxlabel('Time (UTC)')
repr_sounding_fig.suptitle('Continental Representative Profile\n2022-07-14')
repr_sounding_fig.tight_layout()
repr_sounding_fig.savefig(f'./thesis_figs/continental_representative.pdf')